# AtomicNG — Quickstart

**Atomic N-gram Memory**: forward-only запись знаний в n-gram-память LLM
(метод статьи [arXiv:2610.31125](https://arxiv.org/abs/2610.31125)).
`T = T0 + Δ` — базовая таблица заморожена, новое знание копится в разреженном патче Δ;
очистка Δ возвращает исходную модель **побитово**.

Ядро — чистый NumPy (CPU, без ML-фреймворков); torch нужен только для writer'ов и HF.

In [ ]:
# Установка (в репо): pip install -e ".[dev]"  ·  когда выйдет на PyPI: pip install atomicng
import atomicng
print(atomicng.__version__)

## 1. Базовый цикл: запись → чтение → побитовый откат

In [ ]:
model = atomicng.TinyModel()  # CI-бэкенд; для настоящих моделей: atomicng.backends.hf.attach_sidecar
undo = atomicng.UndoJournal()             # точный откат блока (pre-image журнал)
audit = atomicng.AuditLog("audit.jsonl")  # append-only журнал операций

report = atomicng.write(
    model,
    "AtomicNG хранит новые знания в разреженном патче Δ поверх замороженной таблицы.",
    audit=audit,
    undo=undo,
    source="demo-1",
)
print(report.summary())
print(model.memory.stats())

# Проверка неразрушаемости: пустой Δ => выходы как у базы, restore точен
verdict = atomicng.verify_deletion(model)
print(verdict.message)

## 2. Guards: бюджеты и капы (P0)

In [ ]:
policies = atomicng.WritePolicies(row_budget=5, hot_row_norm_cap=1.0)
model2 = atomicng.TinyModel()
try:
    for i in range(10):
        atomicng.write(model2, "document %d about row budgets and guards" % i, policies=policies)
except atomicng.BudgetExceeded as exc:
    print("бюджет сработал атомарно:", exc)

## 3. Протоколы статьи: acquisition / distraction / chain

In [ ]:
docs = ["Document %d about topic-%d with specific facts and numbers %d." % (i, i, i) for i in range(12)]
model3 = atomicng.TinyModel()
suite = atomicng.run_suite(model3, docs, chain_steps=8)
print("acquisition:", suite["acquisition"])
print("chain: drift", suite["chain"]["final_drift"], "recall", suite["chain"]["final_recall"])

## 4. Мониторинг и публикация

In [ ]:
# Safety-gate: verify + drift + hot-rows (публикация только при зелёном)
gate = atomicng.publish_check(model3)
print("publish:", gate.passed, gate.checks)

# Self-contained HTML-отчёт (можно показать в iframe или скачать)
html_report = atomicng.build_report(model3.memory)
print("отчёт:", len(html_report), "байт")

## Далее
- Соответствие статье — `FIDELITY.md`, conformance-тесты: `pytest tests/test_fidelity.py`
- Настоящие модели: `atomicng.backends.hf.attach_sidecar` + `scripts/train_writer.py`
- CLI: `atomicng write / eval / bench / monitor / export-ple / publish / smoke`